# POC 2: τ²-bench retail baseline (multi-request tasks)

Pair case [#58](https://github.com/zhongnz/Fall26VIP_Agentic_Risk/issues/58), task [#62](https://github.com/zhongnz/Fall26VIP_Agentic_Risk/issues/62).

**Purpose.** Run a small baseline, without any completeness check, on τ²-bench retail tasks that
need two or more state-changing actions, and see whether the agent misses requested changes.
This is a feasibility run: a few tasks and one trial show whether the setup works, not how often
failures happen.

**Before running**
1. Colab secrets (key icon on the left): a secret named `GEMINI_API_KEY`, with **Notebook access** turned on.
2. Runtime → Run all. The models list printed in step 2 shows which Gemini models this key can use.

**Deviation from the original benchmark:** checks of the form "the agent told the user X" are graded by an
LLM judge. τ²-bench's default judge is OpenAI gpt-4.1; here it is the same Gemini model as the agent, asked
for plain JSON, and a code fence around its JSON is removed before parsing.

In [ ]:
# Settings
AGENT_LLM = "gemini/gemini-3.8-flash"   # LiteLLM model name for the agent (2.5-flash is closed to new users)
USER_LLM = "gemini/gemini-3.8-flash"    # LiteLLM model name for the simulated customer
JUDGE_LLM = AGENT_LLM   # grades "agent told the user X" checks; τ²-bench's default judge is OpenAI gpt-4.1
NUM_TASKS = 5
NUM_TRIALS = 1
MAX_CONCURRENCY = 1      # one simulation at a time, to stay within free-tier rate limits
SEED = 300               # τ²-bench default
TAU2_COMMIT = "5bfa7e37b366"

## 1. Install τ²-bench (pinned commit)

In [ ]:
import os, sys, json, subprocess
print(sys.version)
assert (3, 12) <= sys.version_info[:2] < (3, 14), "τ²-bench needs Python 3.12 or 3.13"

if not os.path.exists("tau2-bench"):
    subprocess.run(["git", "clone", "-q", "https://github.com/sierra-research/tau2-bench.git"], check=True)
subprocess.run(["git", "-C", "tau2-bench", "checkout", "-q", TAU2_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "tau2-bench"], check=True)

# The judge model is a constant in τ²-bench's config (no CLI flag); point it at JUDGE_LLM,
# ask it for plain JSON, and strip a ```json fence if one still comes back (Gemini adds them).
import re
cfg_path = "tau2-bench/src/tau2/config.py"
cfg = open(cfg_path).read()
cfg = re.sub(r"^DEFAULT_LLM_NL_ASSERTIONS = .*$", f'DEFAULT_LLM_NL_ASSERTIONS = "{JUDGE_LLM}"', cfg, flags=re.M)
cfg = re.sub(r"^DEFAULT_LLM_NL_ASSERTIONS_ARGS = .*$",
             'DEFAULT_LLM_NL_ASSERTIONS_ARGS = {"temperature": DEFAULT_LLM_NL_ASSERTIONS_TEMPERATURE, '
             '"response_format": {"type": "json_object"}}', cfg, flags=re.M)
open(cfg_path, "w").write(cfg)
ev_path = "tau2-bench/src/tau2/evaluator/evaluator_nl_assertions.py"
ev = open(ev_path).read()
ev = ev.replace("result_data = json.loads(assistant_message.content)",
                'result_data = json.loads(assistant_message.content.strip().removeprefix("```json")'
                '.removeprefix("```").removesuffix("```").strip())')
open(ev_path, "w").write(ev)
print("NL-assertion judge:", JUDGE_LLM)
print("installed tau2 at", subprocess.run(["git", "-C", "tau2-bench", "rev-parse", "--short=12", "HEAD"],
                                          capture_output=True, text=True).stdout.strip())

## 2. API key (from Colab secrets; never printed)

In [ ]:
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

import urllib.request
req = urllib.request.Request("https://generativelanguage.googleapis.com/v1beta/models?pageSize=200",
                             headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})
models = json.load(urllib.request.urlopen(req))["models"]
print("models this key can use:")
print(sorted(m["name"].split("/")[-1] for m in models if "generateContent" in m.get("supportedGenerationMethods", [])))

## 3. Choose tasks

Tasks whose reference solution has two or more state-changing actions (exchanges, returns,
modifications, cancellations), so that a missed request is possible. For this feasibility run we
take the first `NUM_TASKS` by task ID; the selection rule for the real comparison is decided later.

In [ ]:
TASKS_PATH = "tau2-bench/data/tau2/domains/retail/tasks.json"
READ_ONLY = ("get_", "find_", "list_", "calculate", "think", "transfer_to_human_agents")

tasks = json.load(open(TASKS_PATH))
by_id = {t["id"]: t for t in tasks}

def expected_writes(task):
    actions = (task.get("evaluation_criteria") or {}).get("actions") or []
    return [a for a in actions if not a["name"].startswith(READ_ONLY)]

multi = sorted((t for t in tasks if len(expected_writes(t)) >= 2), key=lambda t: int(t["id"]))
print(f"{len(multi)} of {len(tasks)} tasks have two or more state-changing actions")
TASK_IDS = [t["id"] for t in multi[:NUM_TASKS]]
for t in multi[:NUM_TASKS]:
    reason = t["user_scenario"]["instructions"]["reason_for_call"]
    print(f"task {t['id']}: {[a['name'] for a in expected_writes(t)]}\n   {reason[:200]}")

## 4. Run the baseline

In [ ]:
import re
from datetime import datetime, timezone
KEY_PATTERN = re.compile(r"key=[A-Za-z0-9._\-]+")  # error logs can include the request URL with the API key
RUN_NAME = "baseline-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
cmd = ["tau2", "run", "--domain", "retail",
       "--agent-llm", AGENT_LLM, "--user-llm", USER_LLM,
       "--task-ids", *TASK_IDS,
       "--num-trials", str(NUM_TRIALS), "--max-concurrency", str(MAX_CONCURRENCY),
       "--seed", str(SEED), "--save-to", RUN_NAME]
print(" ".join(cmd), "\n")
proc = subprocess.Popen(cmd, cwd="tau2-bench", stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        stdin=subprocess.DEVNULL, text=True, env={**os.environ, "TERM": "dumb"})
for line in proc.stdout:
    print(KEY_PATTERN.sub("key=[REDACTED]", line), end="")
print("\nexit code:", proc.wait())

## 5. Results

For each task: reward (1 = solved under τ²-bench scoring), whether the final database matches,
and which expected state-changing actions the agent never made (a candidate missed request).
The agent's last message is printed for failed tasks, to check later whether it claimed completion.

In [ ]:
RESULTS = f"tau2-bench/data/simulations/{RUN_NAME}/results.json"
res = json.load(open(RESULTS))
sims = res["simulations"]

def same_call(call, action):
    return call["name"] == action["name"] and call.get("arguments") == action.get("arguments")

rows = []
for s in sims:
    task = by_id[s["task_id"]]
    msgs = s.get("messages") or []
    calls = [tc for m in msgs if m.get("role") == "assistant" for tc in (m.get("tool_calls") or [])]
    expected = expected_writes(task)
    missing = [a["name"] for a in expected if not any(same_call(c, a) for c in calls)]
    last_agent = next((m.get("content") for m in reversed(msgs) if m.get("role") == "assistant" and m.get("content")), "")
    ri = s.get("reward_info") or {}
    rows.append({
        "task_id": s["task_id"], "trial": s.get("trial"),
        "reward": ri.get("reward"), "db_match": (ri.get("db_check") or {}).get("db_match"),
        "expected_writes": len(expected), "missing_writes": missing,
        "agent_tool_calls": len(calls), "agent_turns": sum(m.get("role") == "assistant" for m in msgs),
        "termination": s.get("termination_reason"), "seconds": round(s.get("duration") or 0, 1),
        "agent_cost": s.get("agent_cost"), "user_cost": s.get("user_cost"),
        "last_agent_message": last_agent,
    })

n = len(rows)
print(f"solved: {sum((r['reward'] or 0) >= 1 for r in rows)}/{n}")
print(f"tasks with a missing expected change: {sum(bool(r['missing_writes']) for r in rows)}/{n}")
print(f"agent tool calls: {sum(r['agent_tool_calls'] for r in rows)} in total\n")
for r in rows:
    print(f"task {r['task_id']}: reward={r['reward']} db_match={r['db_match']} "
          f"missing={r['missing_writes']} tool_calls={r['agent_tool_calls']} end={r['termination']}")
    if (r["reward"] or 0) < 1:
        print("   last agent message:", (r["last_agent_message"] or "")[:400].replace("\n", " "))

## 6. Save and download

In [ ]:
summary_path = f"summary-{RUN_NAME}.json"
with open(summary_path, "w") as f:
    json.dump({"run": RUN_NAME, "tau2_commit": TAU2_COMMIT, "agent_llm": AGENT_LLM, "user_llm": USER_LLM, "judge_llm": JUDGE_LLM,
               "task_ids": TASK_IDS, "num_trials": NUM_TRIALS, "seed": SEED, "rows": rows}, f, indent=2)
results_copy = f"results-{RUN_NAME}.json"
with open(RESULTS) as src, open(results_copy, "w") as dst:
    dst.write(KEY_PATTERN.sub("key=[REDACTED]", src.read()))

from google.colab import files
files.download(summary_path)
files.download(results_copy)